# Benchmarking Conditions + Evaluation Framework

## Goals: 
1. Define a standardized framework for evaluating and comparing ML models for BuyWise.
    * Defining how to compare the models fairly
    * Define dataset selection
    * Establishing evaluation metrics for price prediction and BUY/WAIT reccomendations
    * Setting latency and cost benchmarks

## Problem 

1. Price Forecasting (Regression)
* **Goal**: Predict future product prices over a given period of time (7 day & 14 day forecasts)
    * Given historical price data and product features, the model should output predicted future price. 
        * Input: historical price time series & product features
        * Output: predicted future price
2. Recommendation Generation (Classification)
* **Goal**: Convert predicted price changes into reccomendations (BUY or WAIT). 
    * A classification decision is made based on whether the predicted price drop exceeds a predfined threshold. 
        * Input: predicted price, current price
        * Output: BUY or WAIT reccomendations
3. Confidence Estimation (Probability Prediction)
* **Goal**: Estimate the probability that the predicted price drop/rise will occur. 
    * The probability serves as a confidence score shown to users, indicating how reliable the prediction is.
        * Input: predicted price drop/rise & historical features
        * Output: probability/confidence score



# Dataset Selection + Test/Train Strategy

The benchmarking framework will use historical product price data obtained from the Keepa API. 

### Dataset Scope
* The initial dataset will consist of approximately **300-500** products in the **electronics category**.
* Each product will have historical price time-series data. 
* **Data Granularity**: daily price observations (subject to API format)

### Reasoning
* The dataset size is chosen to
    * Ensure feasibility for local computation
    * Provide sufficient diversity across products
    * Capture real-world retail pricing behavior


**This dataset will be consistent across all models to ensure a fair comparison.**

### Train/Test Split Strategy
Since product pricing is a time-series problem, we will use a time-based split rather than a random split. 

### Approach
* Training Set: historical data up to time T
* Test Set: future data after time T

This will ensure that models are evaluated on their ability to predict unseen future prices. Models must predict future prices based only on past data. 

### Prediction Horizons
The models will be evaluated based on their ability to predict price changes over multiple time periods (7 day or 14 day).
* Short-term: 7-day forecast
* Long-term: 14-day forecast

Each model will generate predictions for both of the horizons, and their performance will be evaluated seperately for each. 

### Reasoning
Different time horizons will capture different pricing behaviors (ex: short term fluctuations, promotional cycles, etc).

### Benchmarking Consistency 

To ensure that there is a fair comparison across models: 
* All models will be trained on the same dataset. 
* All models will use the same train/test split. 
* All models will predict the same time horizons. 

This will ensure that performance differences are due to the capabilities of the model instead of differences in the set up or data. 


# Evaluation Metrics

To evaluate the accuracy of predicted future prices, we will use the following regression metrics. 

### Mean Absolute Error (MAE)
Measures the average absolute difference between predicted and actual prices
* Less sensitive to large outliers
* Treats all errors equally, easily interpretable

### Root Mean Squared Error (RMSE)
Measures the square root of the average squared error.
* Maintains a strong penalty for large errors while converting the result back to the original unit of the target variable, improves interpretability  

### R^2 Score (Coefficient of Determination)
Measures how well the model explains variance in price data

* 1.0 - indicates perfect price prediction
* 0 - indicates no predictive power

Provides insight into how well the model captures underlying data patterns

### Reasoning
These metrics will provide a comprehensive view of prediction accuracy and are standard metrics in time-series forecasting problems. 

## Classification Metrics (BUY or WAIT)
To evaluate the quality of BUY/WAIT reccomendations, we will use classification metrics derived from the confusion matrix. 

### Confusion Matrix
The confusion matrix categorizes predictions into: 
- True Positive (TP): Correct WAIT prediction (price drops as expected)
- False Positive (FP): Incorrect WAIT (price does not drop)
- False Negative (FN): Missed opportunity (price drops but model says BUY)
- True Negative (TN): Correct BUY decision

### Accuracy
Percentage of correct predictions overall

### Precision
Measures how often WAIT reccomendations are correct (important to avoid unecessary waiting)

**Prediction = TP/ (TP + FP)**

### Recall
Measures how often the model correctly idenitifies price drops (important to capture savings opportunities)

**Recall = TP/ (TP + FN)**

### F1 Score
Harmonic mean of precision and recall

**F1 = 2 * (Precision * Recall) / (Precision + Recall)**

### Reasoning
The F1 score is important when classes are imbalanced. For example, when there are fewer price drops than stable prices. 

Note on thresholds: The The classification decision depends on a threshold (ex: minimum percentage price drop). Adjusting this threshold will affect the tradeoff between false positives and false negatives.


## Confidence/Probability Evaluation 
The system will also output a probability/confidene score that indicates how likely the predicted price drop/rise is to occur. 

### Calibration
A well calibrated model will produce probabilitites that reflect true likelihoods.

### Evaluation Approach 
We evaluate how well predicted probabilities align with actual outcomes by comparing predicted confidence levels with observed frequencies.

### Reasoning
The confidence scores will be shown directly to the users, making calibration important for decision-making. 


## Latency and Efficiency Metrics
To ensure that the models are suitable for real-time deployment, we will evaluate the following. 

### Inference Latency
Time taken to generate a prediction for a single input

Target: 
* < 200 ms for model inference
* < 500 ms total response time

### Throughput
Number of predictions processed per second (for batch scenarios)

### Reasoning
Low latency is important for user experience because predictions have to be delivered instantly in the browser extension. 

## Cost per Inference
We will estimate the computational cost of generating predictions. 
 
**Cost per Inference = total compute cost/number of predictions**

Factors we will consider:
* Model complexity
* Hardware requirements (CPU vs GPU)
* Use of caching

### Reasoning
Efficient models will reduce operatonal costs and improve scalability. 

## Batch Size Assumptions

We have two inference scenarios:

### 1. Real-Time Inference (Primary Case)
* Batch Size = 1
* Represents a single user request from the browser extension 

### 2. Batched Inference (Secondary Case)
* Batch Size = 10-100
* Represents the backend processing or precomputing predictions

### Priority
All the latency benchmarks will prioritze batch size = 1 to reflect the real-time system requirements. 

## Fair Comparison Conditions
To ensure that all the models are evaluated fairly:
* The same dataset must be used for all models. 
* The same train/test split strategy is applied. 
* The same prediction horizons are used. 
* The same evaluation metrics are used. 

## Inference Conditions
The models will be evaluated under the following conditions that will simulate real-world deployment. 
* Predictions are generated only using past data. 
* There is no access to future information during inference. 
* The models must produce predictions within latency constraints. 

## Model Selection Constraints
The models will be evaluated based on the following: 
* Accuracy (prediction and classification performance)
* Latency (ability to meet real-time requirements)
* Efficiency (compute cost and scalability)
* Relevance to structured retail price data

The models that are too slow or computationally expensive for real-time use will be deprioritized, even if they have a higher accuracy. 

## Reproductability
To ensure that there are consistent and repeatable results: 
* All experiments should use fixed, random seeds where they are applicable
* Data preprocessing steps should be documented
* Model configurations and hyperparameters should be recorded



# Price and Non-Price Factors

## Price-Based Features
These features would be derived from historical price time-series data. Can be included in the "Find Out More" section of extension

### Lagged Prices
 * Previous prices at different time steps (t-1, t-7, t-14)
 * Capture short-term and medium-term trends

#### Moving Averages
 * Rolling averages over windows (7-day, 14-day)
 * Smooth out noise and highlight trends

#### Price Change Rate
* Percentage change over time
* Helps identify momentum or rapid drops

#### Price Volatility
* Standard deviation of price over a window
* Indicates how unstable pricing is

#### Historical Minimum/Maximum Prices
 * Helps identify whether current price is relatively high or low

### Reasoning
These features will capture patterns such as trends, seasonality, and short-term fluctuations in pricing.


## Non-Price Features
These features will provide additional context beyond price history and improve predictive performance.

### Product Metadata
 * Category (electronics, accessories, etc.)
 * Brand
 * Release date

 Reasoning: Different product categories will have different pricing behaviors. For example, electronics depreciate quickly after release. 

### Demand Signals
* Review count
* Product rating
* Sales rank

Reasoning: High-demand products tend to maintain stable prices, while low-demand items will experience more frequent discounts.

### Availability / Inventory Signals
* In-stock vs out-of-stock status
* Warehouse deals or resale options

Reasoning: Low inventory or high demand may prevent price drops, while excess inventory can lead to discounts.

### Promotional Signals
* Lightning deals
* Discounts
* Special offers

Reasoning: Promotions directly affect short-term price behavior and often precede price drops.

### Seasonal / Time Features
* Month
* Week of year
* Days until major events (Black Friday, Prime Day)

Reasoning: Retail prices follow strong seasonal patterns which are tied to sales events and holidays.

### Competitive Factors
* Number of sellers
* Price differences across sellers

Reasoning: Higher competition often leads to price reductions.

### Category Importance
* Example:
    * Electronics: release date and seasonal events are highly influential
    * Everyday items: demand and inventory may play a larger role

Reasoning: Different features can contribute differently depending on the product category. 

### External Influences/Market Signals (Macroeconomics and Current Events)
* Company stock price trends 
* Supply chain disruptions 
* Macroeconomic indicators
* Major news or events affecting a company or product category

Reasoning: External factors can signficantly influence product pricing. For example, a drop in a company's stock price can signal reduced demand or upcoming discounts. Supply chain disruptions can lead to prices increasing due to limited availability. Economic conditions like inflation can shift pricing overall. 

# References
1. Google ML Crash Course - Classfication Thresholds and Confusion Matrix
2. Keepa API Documentation
3. https://medium.com/@manrajchalokia/time-based-splitting-and-determining-if-train-test-data-come-from-the-same-distribution-e1d2ea881af8
4. https://www.geeksforgeeks.org/machine-learning/regression-metrics/
5. https://www.geeksforgeeks.org/machine-learning/metrics-for-machine-learning-model/
6. https://docs.anyscale.com/llm/serving/benchmarking/metrics
7. https://introl.com/blog/inference-unit-economics-true-cost-per-million-tokens-guide
8. https://medium.com/@gunkurnia/understanding-and-determining-the-ideal-batch-size-in-machine-learning-d3ef93acf79a
9. https://sell.amazon.com/blog/amazon-pricing-strategies
10. https://www.boardfy.com/a-guide-to-amazons-pricing-strategy/

